In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Cài đặt tất cả thư viện cần thiết cho toàn bộ pipeline
!pip install -q sentence-transformers chromadb rank_bm25
!pip install -q transformers accelerate bitsandbytes
!pip install -q FlagEmbedding          # bge-reranker-base (Cross-Encoder)
!pip install -q ragas datasets openai  # Ragas evaluation framework
!pip install -q rouge-score nltk       # ROUGE-L / BLEU
!pip install -q scikit-learn
print("✅ Đã cài đặt xong tất cả thư viện.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

In [ ]:
import os, json, torch, numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm import tqdm

# ── Đường dẫn ──────────────────────────────────────────────────────────────
ROOT_DIR       = "/content/drive/MyDrive/nutrition_rag"
PROCESSED_DIR  = os.path.join(ROOT_DIR, "data_processed")
VECTOR_DB_DIR  = os.path.join(ROOT_DIR, "vector_db")
ALL_RECORDS_FILE = os.path.join(PROCESSED_DIR, "all_records.json")
OUTPUT_DIR     = os.path.join(ROOT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load Knowledge Base ────────────────────────────────────────────────────
with open(ALL_RECORDS_FILE, "r", encoding="utf-8") as f:
    all_records = json.load(f)
print(f"Đã load {len(all_records)} records từ all_records.json")

ids       = [r["record_id"]          for r in all_records]
documents = [r["text_for_embedding"] for r in all_records]

# ── Embedding Model: BAAI/bge-m3 ──────────────────────────────────────────
print("Đang tải BAAI/bge-m3 ...")
embed_model = SentenceTransformer("BAAI/bge-m3")

# ── ChromaDB ──────────────────────────────────────────────────────────────
client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = client.get_or_create_collection(
    name="nutrition_collection",
    metadata={"hnsw:space": "cosine"}
)

existing_count = collection.count()
if existing_count > 0:
    print(f"ChromaDB đã có {existing_count} bản ghi → Bỏ qua re-embedding.")
else:
    print("ChromaDB trống → Bắt đầu embedding ...")
    metadatas = []
    for record in all_records:
        meta = {
            "record_type": record.get("record_type", ""),
            "source":      record.get("source", ""),
        }
        for key in ("group", "age", "food_name_en"):
            if record.get(key):
                meta[key] = record[key]
        metadatas.append(meta)

    BATCH = 100
    for i in tqdm(range(0, len(ids), BATCH)):
        s, e = i, i + BATCH
        batch_embeddings = embed_model.encode(documents[s:e], convert_to_tensor=False).tolist()
        collection.upsert(
            ids=ids[s:e],
            documents=documents[s:e],
            embeddings=batch_embeddings,
            metadatas=metadatas[s:e]
        )
    print(f"✅ Đã tạo {collection.count()} vectors vào ChromaDB.")

# ── BM25 Index ────────────────────────────────────────────────────────────
print("Đang khởi tạo BM25 Index ...")
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)
print("✅ BM25 Index sẵn sàng.")
